# AWQ-opt2.7B GPTQ量化

## 初始化

In [1]:
import gc
import os
import time
from pathlib import Path

os.environ.setdefault(
    "PYTORCH_CUDA_ALLOC_CONF",
    "expandable_segments:True",
)

import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from gptqmodel import GPTQModel, BACKEND
from gptqmodel.quantization import AWQConfig, FORMAT
from gptqmodel.nn_modules.qlinear.torch_awq import (
    AwqTorchLinear,
)

/home/sigan/miniconda3/envs/nowLLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


W1106 10:14:58.479000 81248 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1106 10:14:58.588000 81248 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.7.0
Transformers : 5.20.2
Torch        : 2.14.1+cu130
Triton       : 3.8.0


In [2]:
model_id = "facebook/opt-2.7b"

quant_path = Path(
    "/sigan/ai-ml/models/quantized/"
    "opt-2.7b-awq-4bit-GPTQ"
)

quant_path.mkdir(
    parents=True,
    exist_ok=True,
)

print("模型：", model_id)
print("输出：", quant_path)

模型： facebook/opt-2.7b
输出： /sigan/ai-ml/models/quantized/opt-2.7b-awq-4bit-GPTQ


## 校准数据

In [3]:
wiki_dataset = load_dataset(
    "wikitext",
    "wikitext-2-raw-v1",
    split="train",
)

calibration_dataset = [
    text.strip()
    for text in wiki_dataset["text"]
    if len(text.strip()) > 100
][:128]

print("校准样本数：", len(calibration_dataset))
print(
    "第一条样本：",
    calibration_dataset[0][:300],
)

INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/Salesforce/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/api/resolve-cache/datasets/Salesforce/wikitext/b08601e04326c79dfdd32d625aee71d232d685c3/README.md?%2Fdatasets%2FSalesforce%2Fwikitext%2Fresolve%2Fmain%2FREADME.md=&etag=%222a4fec2bc8df76c9d4da1c8e8865b625eb221c76%22 "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 307 Temporary Redirect"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/datasets/Salesforce/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 404 Not Found"
INFO:httpx2:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/wikitext/wikit

校准样本数： 128
第一条样本： Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in Janua


## 创建AWQ配置

In [4]:
quant_config = AWQConfig(
    bits=4,
    group_size=128,
    sym=False,
    format=FORMAT.GEMM,
    scale_search_chunked_activations=True,
    offload_to_disk=False,
)

## 加载原模型并量化

In [5]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"

In [6]:
torch.cuda.reset_peak_memory_stats()

model = GPTQModel.load(
    model_id,
    quantize_config=quant_config,
    dtype=torch.float16,
    backend=BACKEND.AWQ_TORCH,
)

print("model_type：", model.model.config.model_type)

INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b/revision/main "HTTP/1.1 200 OK"
Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 1627.08it/s]
INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]
INFO  Loader: loading model directly to CPU (not using meta device or turtle_model)


Loading weights: 100%|██████████| 517/517 [00:00<00:00, 12928.40it/s]
INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "eos_token_id": 2,
  "pad_token_id": 1
}

INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "do_sample": true,
  "eos_token_id": 2,
  "pad_token_id": 1
}

INFO  Kernel: loaded -> `[]`                                                    
model_type： opt


## 开始量化

In [7]:
print(model.qlinear_kernel)

None


In [8]:
import time
import torch

start = time.perf_counter()

model.quantize(
    calibration_dataset,
    batch_size=1,
)

elapsed = time.perf_counter() - start

print(f"量化耗时：{elapsed / 60:.2f} 分钟")

INFO  Packing Kernel: selected: `AwqGEMMLinear`                                 
INFO  Packing Kernel: selected: `AwqGEMMLinear`                                 
WARN  Calibration dataset size should be more than 256. Current: 128.           
INFO  Calibration: Sort in descending order by length                           
INFO  Calibration: Total padded tokens: 0                                       
INFO  Calibration: Total non-padded tokens: 19406                               
INFO  Calibration: Total tokens: 19406                                          
WARN  The average length of input_ids of calibration_dataset should be greater than 256: actual avg: 151.609375.
INFO  ModuleLooper: capturing layer inputs from 128 calibration batches         
INFO  +------------+-------+--------+-------+---------+--------+---------+      
INFO  | region     | count | last_s | avg_s | total_s | pct    | source  |      
INFO  +------------+-------+--------+-------+---------+--------+---------+   

AWQProcessor: layer 0 fallback quant 6 modules: ['self_attn.q_proj', 'fc2', 'self_attn.out_proj', 'self_attn.v_proj', 'self_attn.k_proj', 'fc1']


INFO  +-+--+--+--+---+-+-+-+-+-+--+-+-----+------+---+----+--+----+-----+--+---+
INFO  |p|la|mo|fe|...|l|s|d|t|f|(v|d|ac...|act...|...|a...|sc|s...|sc...|sc|...|
INFO  +-+--+--+--+---+-+-+-+-+-+--+-+-----+------+---+----+--+----+-----+--+---+
INFO  |a|0|se|2...|f...|n|...| |0| |cu| |...|22|1|lat...|se...|...|22 |1|lat...|
INFO  +-+-+--+----+----+-+---+-+-+-+--+-+---+--+-+------+-----+---+---+-+------+
INFO  |a|0|f|10...|f1...|...|19| |...| |c...| |19|22|1|late...|fc|19|2|1|lat...|
INFO  +-+-+-+-----+-----+---+--+-+---+-+----+-+--+--+-+-------+--+--+-+-+------+
INFO  |aw|0|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|0|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|0|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-

AWQProcessor: layer 1 fallback quant 6 modules: ['self_attn.q_proj', 'fc2', 'self_attn.out_proj', 'self_attn.v_proj', 'self_attn.k_proj', 'fc1']


INFO  |p|la|mo|fe|...|l|s|d|t|f|(v|d|ac...|act...|...|a...|sc|s...|sc...|sc|...|
INFO  +-+--+--+--+---+-+-+-+-+-+--+-+-----+------+---+----+--+----+-----+--+---+
INFO  |a|1|se|2...|f...|n|...| |0| |cu| |...|22|1|lat...|se...|...|22 |1|lat...|
INFO  +-+-+--+----+----+-+---+-+-+-+--+-+---+--+-+------+-----+---+---+-+------+
INFO  |a|1|f|10...|f1...|...|19| |...| |c...| |19|22|1|late...|fc|19|2|1|lat...|
INFO  +-+-+-+-----+-----+---+--+-+---+-+----+-+--+--+-+-------+--+--+-+-+------+
INFO  |aw|1|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|1|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|1|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |a|1|f|25...|f1...|...

AWQProcessor: layer 2 fallback quant 6 modules: ['self_attn.q_proj', 'fc2', 'self_attn.out_proj', 'self_attn.v_proj', 'self_attn.k_proj', 'fc1']


... (55 outputs left out)

AWQProcessor: layer 30 fallback quant 6 modules: ['self_attn.q_proj', 'fc2', 'self_attn.out_proj', 'self_attn.v_proj', 'self_attn.k_proj', 'fc1']


INFO  |p|la|mo|fe|...|l|s|d|t|f|(v|d|ac...|act...|...|a...|sc|s...|sc...|sc|...|
INFO  +-+--+--+--+---+-+-+-+-+-+--+-+-----+------+---+----+--+----+-----+--+---+
INFO  |a|3|se|2...|f...|n|...| |0| |cu| |...|22|1|lat...|se...|...|22 |1|lat...|
INFO  +-+-+--+----+----+-+---+-+-+-+--+-+---+--+-+------+-----+---+---+-+------+
INFO  |a|30|f|10...|f1...|...|19| |...| |c...| |19|22|1|lat...|fc|19|2|1|lat...|
INFO  +-+--+-+-----+-----+---+--+-+---+-+----+-+--+--+-+------+--+--+-+-+------+
INFO  |aw|3|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|3|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|3|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |a|30|f|25...|f1...|..

AWQProcessor: layer 31 fallback quant 6 modules: ['self_attn.q_proj', 'fc2', 'self_attn.out_proj', 'self_attn.v_proj', 'self_attn.k_proj', 'fc1']


INFO  |p|la|mo|fe|...|l|s|d|t|f|(v|d|ac...|act...|...|a...|sc|s...|sc...|sc|...|
INFO  +-+--+--+--+---+-+-+-+-+-+--+-+-----+------+---+----+--+----+-----+--+---+
INFO  |a|3|se|2...|f...|n|...| |0| |cu| |...|22|1|lat...|se...|...|22 |1|lat...|
INFO  +-+-+--+----+----+-+---+-+-+-+--+-+---+--+-+------+-----+---+---+-+------+
INFO  |a|31|f|10...|f1...|...|19| |...| |c...| |19|22|1|lat...|fc|19|2|1|lat...|
INFO  +-+--+-+-----+-----+---+--+-+---+-+----+-+--+--+-+------+--+--+-+-+------+
INFO  |aw|3|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|3|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |aw|3|se|2...|f16...|n|...| |0| |cu| |...|2|1|la...|self...|...|2|1|la...|
INFO  +--+-+--+----+------+-+---+-+-+-+--+-+---+-+-+-----+-------+---+-+-+-----+
INFO  |a|31|f|25...|f1...|..

## 保存模型

In [9]:
model.save(str(quant_path))

tokenizer = AutoTokenizer.from_pretrained(
    model_id
)

tokenizer.save_pretrained(
    str(quant_path)
)

print("AWQ 模型已保存：", quant_path)

Writing model shards: 0it [00:00, ?it/s]

INFO  Saved Quantize Config: 
{
  "bits": 4,
  "group_size": 128,
  "desc_act": false,
  "lm_head": false,
  "method": "awq",
  "quant_method": "awq",
  "format": "gemm",
  "checkpoint_format": "gemm",
  "pack_dtype": "int32",
  "meta": {
    "quantizer": [
      "gptqmodel:7.7.0"
    ],
    "uri": "https://github.com/modelcloud/gptqmodel",
    "damp_percent": null,
    "damp_auto_increment": null,
    "static_groups": null,
    "true_sequential": true,
    "mse": null,
    "gptaq": null,
    "foem": null,
    "act_group_aware": null,
    "fallback": {
      "strategy": "rtn",
      "threshold": "0.5%",
      "smooth": null
    },
    "offload_to_disk": false,
    "telemetry": {
      "device": false
    },
    "offload_to_disk_path": null,
    "pack_impl": "auto",
    "gc_mode": "interval",
    "wait_for_submodule_finalizers": false,
    "auto_forward_data_parallel": true,
    "dense_vram_strategy": "exclusive",
    "dense_vram_strategy_devices": null,
    "moe_vram_strategy": "exclus

Files in directory:
quant_log.csv
config.json
generation_config.json
quantize_config.json
Content of saved `generation_config.json`:
{
    "_from_model_config": true,
    "bos_token_id": 2,
    "do_sample": true,
    "eos_token_id": 2,
    "pad_token_id": 1,
    "transformers_version": "5.20.2"
}
Content of saved `config.json`:
{
    "_remove_final_layer_norm": false,
    "activation_dropout": 0.0,
    "activation_function": "relu",
    "architectures": [
        "OPTForCausalLM"
    ],
    "attention_dropout": 0.0,
    "bos_token_id": 2,
    "do_layer_norm_before": true,
    "dropout": 0.1,
    "dtype": "float16",
    "enable_bias": true,
    "eos_token_id": 2,
    "ffn_dim": 10240,
    "hidden_size": 2560,
    "init_std": 0.02,
    "layer_norm_elementwise_affine": true,
    "layerdrop": 0.0,
    "max_position_embeddings": 2048,
    "model_type": "opt",
    "num_attention_heads": 32,
    "num_hidden_layers": 32,
    "pad_token_id": 1,
    "prefix": "</s>",
    "quantization_config": {

INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/facebook/opt-2.7b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/api/resolve-cache/models/facebook/opt-2.7b/905a4b602cda5c501f1b3a2650a4152680238254/config.json?%2Ffacebook%2Fopt-2.7b%2Fresolve%2Fmain%2Fconfig.json=&etag=%2202bc707d06e86e9aebea913895b6d0a61627d063%22 "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: GET https://hf-mirror.com/api/models/facebook/opt-2.7b "HTTP/1.1 429 Too Many Requests"
INFO:httpx2:HTTP Request: HEAD https://hf-mirror.com/facebook/opt-2.7b/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"


AWQ 模型已保存： /sigan/ai-ml/models/quantized/opt-2.7b-awq-4bit-GPTQ


## 重新加载 AWQ 模型

In [14]:
from gptqmodel import BACKEND, GPTQModel

tokenizer = AutoTokenizer.from_pretrained(
    str(quant_path),
    local_files_only=True,
)

# 加载 AWQ 量化权重，并指定模型里“未被量化”的那部分层（如 embedding、lm_head 等）使用 torch.float16 精度来计算和存储
reloaded_model = GPTQModel.load(
    str(quant_path),
    device="cuda:0",
    dtype=torch.float16,
    backend=BACKEND.AWQ_TORCH,
)

hf_model = reloaded_model.model
hf_model.eval()

print("推理后端：", BACKEND.AWQ_TORCH)

from_quantized: adapter: None
INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  The layer model.decoder.embed_tokens is not quantized.                    
INFO  The layer model.decoder.embed_positions is not quantized.                 
INFO  Kernel: selected: `AwqTorchLinear`                                        
INFO  Kernel: candidates -> `[AwqTorchLinear]`                                  
INFO  Kernel: per-module selections -> `[AwqTorchLinear=192]`                   
INFO  Kernel: selected -> `AwqTorchLinear`.                                     
INFO  Loader: device = cuda:0                                                   
INFO  Loader: honoring explicit device_map request: {'': 'cuda:0'}              
INFO  Loader: device_map = {'': 'cuda:0'}                                       


INFO  Kernel: selected: `AwqTorchLinear`                                        
INFO  gc.collect() reclaimed 10 objects in 0.126s                               


INFO:tokenicer.tokenicer:Tokenicer: Auto fixed pad_token_id=1 (token='<pad>').


INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 1,
  "use_cache": true
}

INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 2,
  "do_sample": true,
  "eos_token_id": 2,
  "pad_token_id": 1
}

INFO  Kernel: loaded -> `[AwqTorchLinear]`                                      
推理后端： BACKEND.AWQ_TORCH


In [11]:
prompt = "Merry Christmas! I'm glad to"

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to("cuda:0")

with torch.no_grad():
    output_ids = hf_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print(
    tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
    )
)

Merry Christmas! I'm glad to see you're still around.
I'm still here, just not posting as much. I'm still here, just not posting as much. I'm still here, just not posting as much.
I'm glad to hear that. I hope you're doing well.


## 检查量化层

In [12]:
q_proj = (
    hf_model.model
    .decoder.layers[0]
    .self_attn.q_proj
)

print("类型：", type(q_proj).__name__)
print("输入维度：", q_proj.in_features)
print("输出维度：", q_proj.out_features)

print(
    "qweight：",
    q_proj.qweight.shape,
    q_proj.qweight.dtype,
)

print(
    "qzeros：",
    q_proj.qzeros.shape,
    q_proj.qzeros.dtype,
)

print(
    "scales：",
    q_proj.scales.shape,
    q_proj.scales.dtype,
)

类型： AwqTorchLinear
输入维度： 2560
输出维度： 2560
qweight： torch.Size([2560, 320]) torch.int32
qzeros： torch.Size([20, 320]) torch.int32
scales： torch.Size([20, 2560]) torch.float16


## 查看模型大小

In [13]:
def directory_size_gb(path):
    return sum(
        file.stat().st_size
        for file in Path(path).rglob("*")
        if file.is_file()
    ) / 1024**3


print(
    "AWQ 模型目录大小 GB：",
    directory_size_gb(quant_path),
)

print(
    "模型张量近似占用 GB：",
    hf_model.get_memory_footprint() / 1024**3,
)

AWQ 模型目录大小 GB： 1.47255914285779
模型张量近似占用 GB： 1.4691352844238281
